# GitHub CLI (gh) — 심층 분석 및 예제

**Repository**: https://github.com/cli/cli  
**Issue**: #10 — GitHub CLI 코드 분석 및 핵심 요약

---

## 개요

GitHub CLI(`gh`)는 터미널에서 GitHub의 모든 기능을 사용할 수 있도록 해주는 공식 커맨드라인 도구입니다.  
Pull Request, Issue, Release, Workflow 등을 브라우저 없이 터미널에서 직접 관리할 수 있습니다.

| 항목 | 내용 |
|------|------|
| 언어 | Go (99.2%) |
| Stars | 43,100+ |
| Forks | 8,077 |
| Contributors | 600+ |
| 최신 버전 | v2.88.1 |
| 지원 플랫폼 | macOS, Windows, Linux |

---

## 1. 아키텍처 분석

### 디렉토리 구조

```
cli/cli/
├── cmd/              # 엔트리포인트 (main.go)
├── pkg/
│   ├── cmd/          # 36개 커맨드 모듈 (issue, pr, repo, ...)
│   ├── iostreams/    # 입출력 스트림 추상화
│   ├── prompt/       # 인터랙티브 프롬프트
│   └── text/         # 텍스트 유틸리티
├── api/              # GitHub API 클라이언트 (REST + GraphQL)
├── internal/
│   ├── authflow/     # OAuth 인증 플로우
│   ├── config/       # 설정 관리
│   └── ghrepo/       # 레포지토리 파싱
├── git/              # Git 명령어 래퍼
└── context/          # 현재 레포/브랜치 컨텍스트
```

### 커맨드 모듈 목록 (pkg/cmd 하위)

| 카테고리 | 커맨드 |
|----------|--------|
| 인증 | `auth`, `ssh-key`, `gpg-key`, `secret` |
| 레포지토리 | `repo`, `release`, `browse` |
| 이슈/PR | `issue`, `pr`, `label`, `project` |
| 개발도구 | `codespace`, `workflow`, `run`, `actions` |
| 유틸리티 | `api`, `alias`, `config`, `search`, `gist`, `extension` |
| 기타 | `attestation`, `cache`, `variable`, `ruleset`, `status` |

## 2. 핵심 설계 패턴

### Factory Pattern — 의존성 주입

```go
// pkg/cmd/factory/default.go
type Factory struct {
    IOStreams        *iostreams.IOStreams
    HttpClient      func() (*http.Client, error)
    GitClient       *git.Client
    Config          func() (config.Config, error)
    BaseRepo        func() (ghrepo.Interface, error)
    Remotes         func() (context.Remotes, error)
    Branch          func() (string, error)
}
```

모든 커맨드는 `Factory`를 통해 HTTP 클라이언트, IO, 설정 등을 주입받아  
테스트 가능성과 모듈성을 극대화합니다.

### Cobra 기반 CLI 구조

```go
// 각 커맨드는 cobra.Command로 정의됨
func NewCmdIssue(f *cmdutil.Factory) *cobra.Command {
    cmd := &cobra.Command{
        Use:   "issue <command>",
        Short: "Manage issues",
    }
    cmd.AddCommand(NewCmdCreate(f))
    cmd.AddCommand(NewCmdList(f))
    cmd.AddCommand(NewCmdView(f))
    cmd.AddCommand(NewCmdClose(f))
    return cmd
}
```

### Dual API — REST + GraphQL

```go
// GraphQL - 복잡한 데이터 조회 (PR, Issue 상세)
var query struct {
    Repository struct {
        PullRequest struct {
            Number int
            Title  string
        } `graphql:"pullRequest(number: $number)"`
    } `graphql:"repository(owner: $owner, name: $name)"`
}

// REST - 단순 작업 (파일 다운로드, webhook 등)
resp, err := client.REST("GET", "repos/{owner}/{repo}/releases", nil)
```

In [1]:
import subprocess
import json
import os
import re

def run_gh(cmd, input_data=None):
    """gh 명령어를 실행하고 결과를 반환합니다."""
    full_cmd = f"gh {cmd}"
    result = subprocess.run(
        full_cmd, shell=True, capture_output=True,
        text=True, timeout=30
    )
    if result.returncode != 0 and result.stderr:
        # ANSI 코드 제거
        clean_err = re.sub(r'\x1b\[[0-9;]*m', '', result.stderr.strip())
        print(f"  ⚠ {clean_err[:200]}")
    # ANSI 코드 제거 후 반환
    return re.sub(r'\x1b\[[0-9;]*m', '', result.stdout.strip())

def safe_json(text):
    """여러 줄의 JSON 객체를 파싱합니다."""
    results = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return results

# gh 설치 확인
version = run_gh("--version")
print("✅ GitHub CLI 설치 확인")
print(version)

✅ GitHub CLI 설치 확인
gh version 2.88.1 (2026-03-12)
https://github.com/cli/cli/releases/tag/v2.88.1


## 3. 인증 (Authentication)

In [2]:
# 현재 인증 상태 확인
print("=" * 50)
print("🔐 인증 상태")
print("=" * 50)
result = subprocess.run("gh auth status", shell=True, capture_output=True, text=True)
status_out = re.sub(r'\x1b\[[0-9;]*m', '', (result.stdout + result.stderr).strip())
print(status_out)

# 인증된 사용자 정보
print("\n" + "=" * 50)
print("👤 현재 사용자")
print("=" * 50)
user_raw = run_gh('api user')
if user_raw:
    try:
        user_data = json.loads(user_raw)
        print(f"  login:        {user_data.get('login')}")
        print(f"  name:         {user_data.get('name')}")
        print(f"  public_repos: {user_data.get('public_repos')}")
        print(f"  followers:    {user_data.get('followers')}")
        print(f"  following:    {user_data.get('following')}")
    except Exception as e:
        print(f"  파싱 오류: {e}")

🔐 인증 상태


github.com
  ✓ Logged in to github.com account hundong2 (keyring)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo', 'workflow'

👤 현재 사용자


  login:        hundong2
  name:         hundong2
  public_repos: 156
  followers:    155
  following:    652


## 4. Repository 관리

In [3]:
# cli/cli 레포 정보 조회 (gh api 사용)
print("=" * 50)
print("📦 cli/cli 레포지토리 정보")
print("=" * 50)

repo_raw = run_gh('api repos/cli/cli')
if repo_raw:
    data = json.loads(repo_raw)
    print(f"  이름:         {data['name']}")
    print(f"  설명:         {data['description']}")
    print(f"  언어:         {data['language']}")
    print(f"  Stars:        {data['stargazers_count']:,}")
    print(f"  Forks:        {data['forks_count']:,}")
    print(f"  Open Issues:  {data['open_issues_count']:,}")
    print(f"  기본 브랜치:  {data['default_branch']}")
    print(f"  Topics:       {', '.join(data.get('topics', []))}")

📦 cli/cli 레포지토리 정보


  이름:         cli
  설명:         GitHub’s official command line tool
  언어:         Go
  Stars:        43,102
  Forks:        8,077
  Open Issues:  929
  기본 브랜치:  trunk
  Topics:       cli, git, github-api-v4, golang


In [4]:
# cli/cli 최신 릴리즈 조회
print("=" * 50)
print("🚀 cli/cli 최신 릴리즈 (5개)")
print("=" * 50)

releases_raw = run_gh('api repos/cli/cli/releases?per_page=5')
if releases_raw:
    releases = json.loads(releases_raw)
    for r in releases:
        print(f"  📌 {r['tag_name']:<15} {r['name'][:40]:<40} ({r['published_at'][:10]})")

🚀 cli/cli 최신 릴리즈 (5개)


  📌 v2.88.1         GitHub CLI 2.88.1                        (2026-03-12)
  📌 v2.88.0         GitHub CLI 2.88.0                        (2026-03-10)
  📌 v2.87.3         GitHub CLI 2.87.3                        (2026-02-23)
  📌 v2.87.2         GitHub CLI 2.87.2                        (2026-02-20)
  📌 v2.87.1         GitHub CLI 2.87.1                        (2026-02-20)


In [5]:
# 내 레포지토리 목록 (최근 5개)
print("=" * 50)
print("📂 내 레포지토리 (최근 5개)")
print("=" * 50)

my_repos_raw = run_gh('repo list --limit 5 --json name,description,visibility,updatedAt')
if my_repos_raw:
    repos = json.loads(my_repos_raw)
    for r in repos:
        vis = "🔒" if r['visibility'] == 'private' else "🌍"
        desc = (r.get('description') or '(설명 없음)')[:50]
        updated = r['updatedAt'][:10]
        print(f"  {vis} {r['name']:<30} [{updated}]")
        print(f"       {desc}")

📂 내 레포지토리 (최근 5개)


  🌍 my-research-app                [2026-03-14]
       research web site in my local pc 
  🌍 automation_reporter            [2026-03-14]
       (설명 없음)
  🌍 news_letter                    [2026-03-14]
       (설명 없음)
  🌍 copilot_answer_set             [2026-03-13]
       copilot answer set for me 
  🌍 Git-Star-Lab                   [2026-03-13]
       (설명 없음)


## 5. Issue 관리

In [6]:
# cli/cli 레포의 오픈 이슈 조회
print("=" * 55)
print("🐛 cli/cli 최신 이슈 (5개)")
print("=" * 55)

issues_raw = run_gh('api "repos/cli/cli/issues?per_page=5&state=open"')
if issues_raw:
    issues = json.loads(issues_raw)
    for iss in issues:
        date = iss['created_at'][:10]
        print(f"  #{iss['number']:4d} [{date}] @{iss['user']['login']}")
        print(f"         {iss['title'][:65]}")

🐛 cli/cli 최신 이슈 (5개)


  #12928 [2026-03-13] @BagToad
         Env var to host trust mapping for token authentication
  #12927 [2026-03-13] @VitaliVinahradski
         `gh repo sync` silently corrupts worktree when synced branch is c
  #12925 [2026-03-13] @david-crespo
         `gh auth login` doesn't poll for success unless you press enter t
  #12922 [2026-03-12] @1cbyc
         fix(browse): treat decimal-only commit hashes as commits, not iss
  #12921 [2026-03-12] @eevmanu
         Support model selection when requesting Copilot code review from 


In [7]:
# 현재 레포의 이슈 (my-research-app)
print("=" * 55)
print("📋 현재 레포 이슈 목록 (my-research-app)")
print("=" * 55)

my_issues_raw = run_gh('issue list --json number,title,state')
if my_issues_raw and my_issues_raw.strip() != '[]':
    my_issues = json.loads(my_issues_raw)
    for iss in my_issues:
        state_icon = "🟢" if iss['state'] == 'OPEN' else "🔴"
        print(f"  {state_icon} #{iss['number']}: {iss['title'][:60]}")
else:
    print("  이슈 없음")

📋 현재 레포 이슈 목록 (my-research-app)


  🟢 #10: Research for github cli


## 6. Pull Request 관리

In [8]:
# cli/cli 레포의 최신 PR 조회
print("=" * 55)
print("🔀 cli/cli 최신 PR (5개)")
print("=" * 55)

prs_raw = run_gh('api "repos/cli/cli/pulls?per_page=5&state=open"')
if prs_raw:
    prs = json.loads(prs_raw)
    for pr in prs:
        date = pr['created_at'][:10]
        draft = "[DRAFT] " if pr['draft'] else ""
        print(f"  PR #{pr['number']:4d} [{date}] @{pr['user']['login']}")
        print(f"         {draft}{pr['title'][:60]}")

🔀 cli/cli 최신 PR (5개)


  PR #12922 [2026-03-12] @1cbyc
         fix(browse): treat decimal-only commit hashes as commits, no
  PR #12919 [2026-03-12] @dependabot[bot]
         chore(deps): bump golang.org/x/crypto from 0.48.0 to 0.49.0
  PR #12918 [2026-03-12] @dependabot[bot]
         chore(deps): bump advanced-security/filter-sarif from 1.0.1 
  PR #12909 [2026-03-11] @BagToad
         Remove `ChecksStatus` slow path (dead code on all supported 
  PR #12906 [2026-03-11] @BagToad
         Don't count cancelled checks as failures in `pr status`


## 7. GitHub Actions / Workflow 관리

In [9]:
# cli/cli의 워크플로우 조회
print("=" * 55)
print("⚙️  cli/cli GitHub Actions 워크플로우")
print("=" * 55)

wf_raw = run_gh('api repos/cli/cli/actions/workflows')
if wf_raw:
    wf_data = json.loads(wf_raw)
    for wf in wf_data.get('workflows', []):
        state_icon = "✅" if wf['state'] == 'active' else "⏸"
        print(f"  {state_icon} {wf['name']}")
        print(f"     Path: {wf['path']}")

⚙️  cli/cli GitHub Actions 워크플로우


  ✅ Unit and Integration Tests
     Path: .github/workflows/go.yml
  ✅ Lint
     Path: .github/workflows/lint.yml
  ✅ Code Scanning
     Path: .github/workflows/codeql.yml
  ✅ Deployment
     Path: .github/workflows/deployment.yml
  ✅ homebrew-bump-debug
     Path: .github/workflows/homebrew-bump.yml
  ✅ Dependabot Updates
     Path: dynamic/dependabot/dependabot-updates
  ✅ Bump Go
     Path: .github/workflows/bump-go.yml
  ✅ Spam Issue Detection
     Path: .github/workflows/detect-spam.yml
  ✅ Go Vulnerability Check
     Path: .github/workflows/govulncheck.yml
  ✅ Copilot coding agent
     Path: dynamic/copilot-swe-agent/copilot
  ✅ Copilot code review
     Path: dynamic/copilot-pull-request-reviewer/copilot-pull-request-reviewer
  ✅ Dependency Graph
     Path: dynamic/dependabot/update-graph
  ✅ Process Discuss Label
     Path: .github/workflows/triage-discussion-label.yml
  ✅ Issue Triaging
     Path: .github/workflows/triage-issues.yml
  ✅ Triage Scheduled Tasks
     Path: .github

## 8. Search 기능

In [10]:
# gh search — 저장소 검색
print("=" * 55)
print("🔍 GitHub CLI 관련 레포 검색")
print("=" * 55)

search_raw = run_gh('search repos "github cli" --limit 5 --json fullName,description,stargazersCount')
if search_raw:
    repos = json.loads(search_raw)
    for repo in repos:
        desc = (repo.get('description') or '(설명 없음)')[:50]
        print(f"  ⭐ {repo['stargazersCount']:>6,}  {repo['fullName']}")
        print(f"              {desc}")

🔍 GitHub CLI 관련 레포 검색


  ⭐  1,207  frogermcs/GithubClient
              Example of Github API client implemented on top of
  ⭐      9  Codecademy/try-github-CLI-off-platform-project
              (설명 없음)
  ⭐    268  piotrmurach/github_cli
              GitHub on your command line. Use your terminal, no
  ⭐    135  ExistOrLive/GithubClient
              Github iOS Client  based on Github REST V3 API and
  ⭐    216  jsmits/github-cli
              A command-line interface to the GitHub Issues API 


## 9. API 직접 호출 — GraphQL 예제

In [11]:
import tempfile

# GraphQL로 cli/cli 레포의 최근 커밋 히스토리 조회
print("=" * 55)
print("📜 cli/cli 최근 커밋 (GraphQL)")
print("=" * 55)

graphql_query = '''query {
  repository(owner: "cli", name: "cli") {
    defaultBranchRef {
      target {
        ... on Commit {
          history(first: 5) {
            nodes {
              messageHeadline
              author {
                name
                date
              }
              abbreviatedOid
            }
          }
        }
      }
    }
  }
}'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.graphql', delete=False) as f:
    f.write(graphql_query)
    query_file = f.name

result_raw = run_gh(f'api graphql -F query=@{query_file}')
os.unlink(query_file)

if result_raw:
    result_data = json.loads(result_raw)
    history = result_data['data']['repository']['defaultBranchRef']['target']['history']['nodes']
    for c in history:
        print(f"  [{c['abbreviatedOid']}] {c['author']['date'][:10]} — {c['author']['name']}")
        print(f"    {c['messageHeadline'][:70]}")

📜 cli/cli 최근 커밋 (GraphQL)


  [37800dd] 2026-03-12 — William Martin
    Merge pull request #12915 from cli/revert-12596-fix/clarify-scope-erro
  [3921788] 2026-03-12 — William Martin
    Revert "fix: clarify scope error while creating issues for projects"
  [4a3f7d9] 2026-03-12 — William Martin
    Merge pull request #12914 from cli/revert-12845-refactor/deduplicate-…
  [d45acae] 2026-03-12 — William Martin
    Revert "refactor: deduplicate scope error handling between api/client…
  [2bf1669] 2026-03-12 — Kynan Ware
    Merge pull request #12911 from cli/kw/deployment-oidc


## 10. 기여자 분석

In [12]:
# cli/cli 상위 기여자 조회
print("=" * 55)
print("👥 cli/cli 상위 기여자 (Top 10)")
print("=" * 55)

contrib_raw = run_gh('api repos/cli/cli/contributors?per_page=10')
if contrib_raw:
    contributors = json.loads(contrib_raw)
    for rank, c in enumerate(contributors, 1):
        bar = '█' * min(c['contributions'] // 200, 20)
        print(f"  #{rank:2d}  @{c['login']:<25} {c['contributions']:5,} commits {bar}")

👥 cli/cli 상위 기여자 (Top 10)


  # 1  @mislav                    2,061 commits ██████████
  # 2  @BagToad                     781 commits ███
  # 3  @williammartin               635 commits ███
  # 4  @babakks                     513 commits ██
  # 5  @vilmibm                     487 commits ██
  # 6  @andyfeller                  485 commits ██
  # 7  @samcoe                      421 commits ██
  # 8  @probablycorey               396 commits █
  # 9  @malancas                    359 commits █
  #10  @josebalius                  319 commits █


## 11. 언어 통계

In [13]:
# cli/cli 언어 통계
print("=" * 55)
print("💻 cli/cli 언어 통계")
print("=" * 55)

langs_raw = run_gh('api repos/cli/cli/languages')
if langs_raw:
    langs = json.loads(langs_raw)
    total = sum(langs.values())
    sorted_langs = sorted(langs.items(), key=lambda x: x[1], reverse=True)
    for lang, bytes_count in sorted_langs:
        pct = bytes_count / total * 100
        bar = '█' * int(pct / 2)
        print(f"  {lang:<15} {pct:5.1f}%  {bar}")

💻 cli/cli 언어 통계


  Go               99.2%  █████████████████████████████████████████████████
  Shell             0.7%  
  Makefile          0.1%  
  PowerShell        0.0%  
  Batchfile         0.0%  


## 12. 핵심 커맨드 치트시트

In [14]:
cheatsheet = {
    "인증": [
        ("gh auth login",               "GitHub 계정 로그인"),
        ("gh auth status",              "현재 인증 상태 확인"),
        ("gh auth logout",              "로그아웃"),
    ],
    "레포지토리": [
        ("gh repo create <name>",       "새 레포 생성"),
        ("gh repo clone <owner>/<repo>","레포 클론"),
        ("gh repo list",                "내 레포 목록"),
        ("gh repo view --web",          "브라우저로 레포 열기"),
        ("gh repo fork",                "레포 포크"),
    ],
    "이슈": [
        ("gh issue create",             "이슈 생성"),
        ("gh issue list",               "이슈 목록"),
        ("gh issue view <number>",      "이슈 상세 보기"),
        ("gh issue close <number>",     "이슈 닫기"),
        ("gh issue comment <number>",   "이슈 댓글 작성"),
    ],
    "PR": [
        ("gh pr create",                "PR 생성"),
        ("gh pr list",                  "PR 목록"),
        ("gh pr checkout <number>",     "PR 브랜치 체크아웃"),
        ("gh pr review --approve",      "PR 승인"),
        ("gh pr merge --squash",        "PR squash 머지"),
        ("gh pr diff <number>",         "PR diff 보기"),
    ],
    "릴리즈": [
        ("gh release create v1.0.0",    "릴리즈 생성"),
        ("gh release list",             "릴리즈 목록"),
        ("gh release download",         "릴리즈 파일 다운로드"),
    ],
    "API": [
        ("gh api /user",                "REST API 직접 호출"),
        ("gh api graphql -f query='...'","GraphQL 쿼리"),
        ("gh api repos/{owner}/{repo}", "레포 API 조회"),
    ],
    "워크플로우": [
        ("gh workflow list",            "워크플로우 목록"),
        ("gh workflow run <name>",      "워크플로우 수동 실행"),
        ("gh run list",                 "실행 기록 조회"),
        ("gh run watch <run-id>",       "실행 실시간 모니터링"),
    ],
}

for category, cmds in cheatsheet.items():
    print(f"\n{'='*58}")
    print(f"  {category}")
    print(f"{'='*58}")
    for cmd, desc in cmds:
        print(f"  {cmd:<42} # {desc}")


  인증
  gh auth login                              # GitHub 계정 로그인
  gh auth status                             # 현재 인증 상태 확인
  gh auth logout                             # 로그아웃

  레포지토리
  gh repo create <name>                      # 새 레포 생성
  gh repo clone <owner>/<repo>               # 레포 클론
  gh repo list                               # 내 레포 목록
  gh repo view --web                         # 브라우저로 레포 열기
  gh repo fork                               # 레포 포크

  이슈
  gh issue create                            # 이슈 생성
  gh issue list                              # 이슈 목록
  gh issue view <number>                     # 이슈 상세 보기
  gh issue close <number>                    # 이슈 닫기
  gh issue comment <number>                  # 이슈 댓글 작성

  PR
  gh pr create                               # PR 생성
  gh pr list                                 # PR 목록
  gh pr checkout <number>                    # PR 브랜치 체크아웃
  gh pr review --approve                     # PR 승인
  gh pr merge --squash               

## 13. 소스코드 핵심 파일 분석

In [15]:
import base64

# main.go 소스 분석 (GitHub API로 가져오기)
print("=" * 60)
print("🔍 main.go (cmd/gh/main.go) 핵심 구조")
print("=" * 60)

main_raw = run_gh('api repos/cli/cli/contents/cmd/gh/main.go')
if main_raw:
    file_data = json.loads(main_raw)
    content_b64 = file_data['content'].replace('\n', '')
    content = base64.b64decode(content_b64).decode('utf-8', errors='replace')
    print(content[:2500])
    print(f"\n... (총 {len(content):,} 바이트)")

🔍 main.go (cmd/gh/main.go) 핵심 구조


package main

import (
	"os"

	"github.com/cli/cli/v2/internal/ghcmd"
)

func main() {
	code := ghcmd.Main()
	os.Exit(int(code))
}


... (총 131 바이트)


In [16]:
# factory/default.go 분석 — 핵심 설계 패턴
print("=" * 60)
print("🏭 Factory Pattern (pkg/cmd/factory/default.go)")
print("=" * 60)

factory_raw = run_gh('api repos/cli/cli/contents/pkg/cmd/factory/default.go')
if factory_raw:
    file_data = json.loads(factory_raw)
    content_b64 = file_data['content'].replace('\n', '')
    content = base64.b64decode(content_b64).decode('utf-8', errors='replace')
    # Factory struct 부분만 추출
    lines = content.split('\n')
    in_section = False
    shown = 0
    for line in lines:
        if 'func BuildFactory' in line or 'type Factory' in line:
            in_section = True
        if in_section:
            print(line)
            shown += 1
            if shown > 60:
                print("...")
                break

🏭 Factory Pattern (pkg/cmd/factory/default.go)


## 14. 종합 요약

---

### GitHub CLI 핵심 포인트

| 영역 | 핵심 내용 |
|------|----------|
| **언어** | Go (99.2%) — 빠른 빌드, 단일 바이너리 배포 |
| **CLI 프레임워크** | [cobra](https://github.com/spf13/cobra) — 계층적 커맨드 구조 |
| **설계 패턴** | Factory Pattern — 모든 커맨드에 의존성 주입 |
| **API** | REST + GraphQL 동시 지원 |
| **인증** | OAuth Device Flow / Personal Access Token |
| **확장성** | Extension 시스템으로 서드파티 확장 지원 |
| **테스트** | 인터페이스 기반 Mock으로 단위/통합 테스트 |

### 핵심 커맨드 실행 흐름

```
gh pr create
  └─ cmd/gh/main.go              # 엔트리포인트
     └─ pkg/cmd/root/root.go     # 루트 커맨드 라우팅
        └─ pkg/cmd/pr/pr.go      # pr 서브커맨드
           └─ pkg/cmd/pr/create/ # create 로직
              ├─ api/queries_pr.go  # GraphQL로 PR 생성
              └─ git/client.go      # 현재 브랜치 정보 수집
```

### GitHub CLI를 써야 하는 이유

1. **개발 흐름 유지** — 브라우저와 터미널 전환 없이 GitHub 작업 가능
2. **스크립팅 친화적** — CI/CD, 자동화 스크립트에 쉽게 통합
3. **강력한 API 래퍼** — `gh api` 로 인증된 API 호출 + `jq` 파이프라인
4. **확장 가능** — Extension으로 커스텀 워크플로우 구축
5. **오픈소스** — 600+ 기여자, 활발한 커뮤니티
6. **보안** — Sigstore 기반 빌드 증명(attestation) 지원